# Heretic Colab セットアップ
このノートブックは Heretic を Google Colab 上で動かします。GPU を選択し、必要な依存をインストールしてからモデルを実行します。

In [ ]:
#@title オプション: Google Drive をマウント（チェックポイント保存のみ）
# Drive に依存したくない場合は False にしてください（デフォルト: False）。
MOUNT_DRIVE = True  #@param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    import os
    print('Google Drive をマウントしています...')
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/heretic'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('Drive を /content/drive にマウントしました。保存先:', DRIVE_ROOT)
else:
    print('Google Drive のマウントをスキップしました。チェックポイントは /content/checkpoints に保存されます（必要に応じて変更してください）。')


In [1]:
# ランタイム確認（GPU が利用可能か確認し、なければ対処法を表示します）
import shutil, sys, textwrap
if shutil.which('nvidia-smi') is None:
    print(textwrap.dedent('''
nvidia-smi が見つかりません。GPU が有効なランタイムを選択してください。
'''))
else:
    # GPU ドライバ情報を表示
    !nvidia-smi

Sat Apr 18 07:39:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# pip を最新化して依存を入れる（CUDA バージョンに合わせて index-url を変更してください）
!pip install pip -q
!pip install uv -q

!uv self version

uv 0.11.7 (x86_64-unknown-linux-gnu)


リポジトリをクローンします。

In [ ]:
#@title Heretic をクローン（フォームで Repo/Branch を指定）
repo = "Akikukeo1/Live-Vision-Narrator"  #@param {type:"string"}
branch = "main"  #@param {type:"string"}
dest = "/content/Live-Vision-Narrator"  #@param {type:"string"}

!git clone --depth 1 --branch {branch} https://github.com/{repo}.git {dest}

# heretic サブディレクトリを指定
HERETIC_PATH = f"{dest}/heretic"
print(f'✓ HERETIC_PATH を設定しました: {HERETIC_PATH}')


依存をインストール・アップデートする必要があります。

In [ ]:
#@title uv で heretic をインストール（必須）
# 注: Clone セルを先に実行してください。

!cd {HERETIC_PATH} && uv venv --clear && uv pip install -e .

print('\n✓ Heretic のインストールが完了しました。')

# すべての依存を CUDA 128 向けに統一
print('\nすべての依存を CUDA 128 向けに同期しています...')
!uv pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache
!uv pip install --upgrade transformers --no-cache
!uv pip install --upgrade bitsandbytes --no-cache

print('\n✓ 依存関係のインストールとアップデートが完了しました。')


In [ ]:
# チェックポイント保存先の設定（Drive をマウントしている場合は Drive に、そうでなければ /content に保存）
import os, shutil
# MOUNT_DRIVE が定義されていなければ False を仮定
try:
    MOUNT_DRIVE
except NameError:
    MOUNT_DRIVE = False

if MOUNT_DRIVE:
    CHECKPOINT_DIR = '/content/drive/MyDrive/heretic/checkpoints'
else:
    CHECKPOINT_DIR = '/content/checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be saved to', CHECKPOINT_DIR)

# 簡単な同期例（必要に応じて手動で実行）
print('To sync checkpoints to Drive later (if MOUNT_DRIVE is True):')
print('  !rsync -av --progress /content/checkpoints/ /content/drive/MyDrive/heretic/checkpoints/')
print('Or use copy:')
print('  !cp -r /content/checkpoints/* /content/drive/MyDrive/heretic/checkpoints/')


In [ ]:
#@title デバッグ: 仮想環境と CUDA ネイティブライブラリの確認
# 仮想環境内で check.py を実行し、bitsandbytes とネイティブライブラリの状況を確認します

!echo "== nvidia-smi の確認 =="
!nvidia-smi || true

!echo "\n== ldconfig の libnvJitLink 確認 =="
!ldconfig -p | grep libnvJitLink || true

!echo "\n== .venv python check.py の実行 =="
!cd {HERETIC_PATH} && .venv/bin/python check.py || true

!echo "\n== .venv pip freeze =="
!cd {HERETIC_PATH} && .venv/bin/pip freeze | sed -n '1,200p' || true

!echo "\n== bitsandbytes の import テスト (.venv) =="
!cd {HERETIC_PATH} && .venv/bin/python -c "import importlib, sys
try:
    import bitsandbytes as bnb
    print('bitsandbytes の import に成功', getattr(bnb, '__file__', ''))
except Exception as e:
    print('bitsandbytes の import に失敗:', e)
    import traceback; traceback.print_exc()


In [ ]:
#@title pexpect をインストール（自動応答用）
# Auto-resume 機能用に pexpect をインストール
!uv pip install pexpect -q
print('✓ pexpect のインストールが完了しました')

## モデルのセットアップと実行
HuggingFace からモデルをダウンロードして、Heretic で実行します。


In [ ]:
#@title Download model from HuggingFace
from huggingface_hub import snapshot_download
import os

model_id = "google/gemma-4-E4B-it"  #@param {type:"string"}
model_dir = "/content/models"  #@param {type:"string"}
HF_TOKEN = ""  #@param {type:"string"}

os.makedirs(model_dir, exist_ok=True)

# 優先順位: @param の入力値 > Colab userdata の HF_TOKEN > 匿名アクセス
hf_token = None
token_source = "匿名"
if HF_TOKEN.strip():
    hf_token = HF_TOKEN.strip()
    token_source = "@param HF_TOKEN"
    print('認証トークン: @param HF_TOKEN を使用します')
else:
    try:
        from google.colab import userdata
        secret = userdata.get('HF_TOKEN')
        if secret and secret.strip():
            hf_token = secret.strip()
            token_source = "google.colab.userdata の HF_TOKEN"
            print('認証トークン: google.colab.userdata の HF_TOKEN を使用します')
        else:
            print('認証トークンが見つからないため、匿名アクセスで続行します')
    except Exception:
        print('google.colab.userdata が利用できないため、匿名アクセスで続行します')

# 最終的に利用される認証状態を表示（トークン値はマスク）
if hf_token:
    token_preview = f"{hf_token[:4]}...{hf_token[-4:]}" if len(hf_token) >= 8 else "(短すぎるためプレビュー不可)"
    print(f'最終認証: トークン利用（取得元: {token_source}、プレビュー: {token_preview}）')
else:
    print('最終認証: 匿名アクセス')

print(f'{model_id} を {model_dir} にダウンロード中...')
model_path = snapshot_download(
    repo_id=model_id,
    cache_dir=model_dir,
    local_dir=f"{model_dir}/{model_id.replace('/', '--')}",
    local_dir_use_symlinks=False,
    token=hf_token,
 )

print(f'✓ モデルをダウンロードしました: {model_path}')
MODEL_PATH = model_path


In [ ]:
#@title ストレージ使用量を確認
import os
import subprocess

# ストレージ容量を確認
result = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
print('ストレージ使用量:')
print(result.stdout)

# モデルのサイズを確認
if 'MODEL_PATH' in globals() and os.path.exists(MODEL_PATH):
    result = subprocess.run(['du', '-sh', MODEL_PATH], capture_output=True, text=True)
    print(f'\nモデルサイズ: {result.stdout}')


In [ ]:
#@title Drive から復元してバックグラウンド同期を開始（チェックポイント自動同期）
import os
import subprocess
import threading
import time

# ローカル保存先（絶対パス）
LOCAL_CKPTS = '/content/checkpoints'
LOCAL_MODELS = '/content/models'
os.makedirs(LOCAL_CKPTS, exist_ok=True)
os.makedirs(LOCAL_MODELS, exist_ok=True)

# Drive 側の保存先
DRIVE_CKPTS = '/content/drive/MyDrive/heretic/checkpoints'
DRIVE_MODELS = '/content/drive/MyDrive/heretic/models'

# 既に Drive マウント済みと仮定（前のセルで実施済み）
# 必要に応じてここで drive.mount() を実行してください

print("=== Drive からチェックポイントを復元します ===")
os.makedirs(DRIVE_CKPTS, exist_ok=True)
subprocess.run(['rsync', '-av', '--progress', f'{DRIVE_CKPTS}/', f'{LOCAL_CKPTS}/'], check=False)

print("=== バックグラウンド同期スレッドを開始します ===")
SYNC_INTERVAL = 60  # 60 秒ごとに同期

def sync_loop():
    while True:
        try:
            subprocess.run(['rsync', '-av', '--progress', f'{LOCAL_CKPTS}/', f'{DRIVE_CKPTS}/'], check=True, timeout=120)
            print(f"[sync] Drive へチェックポイントを同期しました: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        except Exception as e:
            print(f"[sync error] {e}")
        time.sleep(SYNC_INTERVAL)

# デーモンスレッドで同期を開始
sync_thread = threading.Thread(target=sync_loop, daemon=True, name='checkpoint_sync')
sync_thread.start()
print(f"バックグラウンド同期を開始しました（間隔: {SYNC_INTERVAL}秒）")
print(f"ローカルのチェックポイント: {LOCAL_CKPTS}")
print(f"Drive のチェックポイント: {DRIVE_CKPTS}")

In [ ]:
#@title Heretic を実行（pexpect で自動再開）
# 注: モデルダウンロードセルと「Drive から復元してバックグラウンド同期を開始」セルを先に実行してください

import subprocess
import os
import sys
import pexpect

use_quantization = True  #@param {type:"boolean"}
skip_batch_autodetect = True  #@param {type:"boolean"}
fixed_batch_size = 128  #@param {type:"integer", min:1, max:512}
AUTO_RESUME = True  #@param {type:"boolean"}
AUTO_RESUME_TIMEOUT = 300  #@param {type:"integer", min:10, max:3600}
trial_selection = 1  #@param {type:"integer", min:1, max:100}
auto_exit_after_trial = True  #@param {type:"boolean"}

LOCAL_CKPTS = '/content/checkpoints'
DRIVE_CKPTS = '/content/drive/MyDrive/heretic/checkpoints'
LOCAL_EXPORTED_MODEL = '/content/heretic_exported_model'
DRIVE_EXPORTED_MODEL = '/content/drive/MyDrive/heretic/exported_model'

# コマンド構築
cmd = f"cd {HERETIC_PATH} && uv run --no-sync heretic --model {MODEL_PATH} --study-checkpoint-dir {LOCAL_CKPTS}"
if use_quantization:
    cmd += " --quantization bnb_4bit"
if skip_batch_autodetect:
    cmd += f" --batch-size {fixed_batch_size}"

log_file = f"{HERETIC_PATH}/heretic_run.log"

print(f"実行コマンド: {cmd}")
print(f"チェックポイント保存先: {LOCAL_CKPTS}")
print(f"エクスポートモデル（ローカル）: {LOCAL_EXPORTED_MODEL}")
print(f"ログファイル: {log_file}")
print(f"自動再開: {AUTO_RESUME}")
print(f"自動選択 Trial: {trial_selection}")
print(f"Trial 選択後に自動終了: {auto_exit_after_trial}")
print()

class TeeWriter:
    def __init__(self, *writers):
        self.writers = writers

    def write(self, data):
        for writer in self.writers:
            writer.write(data)
            writer.flush()

    def flush(self):
        for writer in self.writers:
            writer.flush()

# pexpect で自動再開・自動選択
if AUTO_RESUME:
    print("=== pexpect で Heretic を開始（自動再開/自動選択）===")
    try:
        with open(log_file, 'a', encoding='utf-8') as log:
            child = pexpect.spawn('/bin/bash', ['-lc', cmd], encoding='utf-8', timeout=None)
            child.logfile_read = TeeWriter(sys.stdout, log)

            # 1. 開始時のプロンプト（Continue / Restart）を待つ
            try:
                child.expect(r'How would you like to proceed\?', timeout=AUTO_RESUME_TIMEOUT)
                print("[auto-resume] 続行プロンプトを検出。'1'（Continue）を送信します...")
                child.sendline('1')
            except (pexpect.TIMEOUT, pexpect.EOF):
                print("[auto-resume] 初期プロンプトは表示されませんでした（通常起動）。")

            # 2. 最適化完了後の「Which trial do you want to use?」を待つ
            try:
                # 非常に長い時間がかかる可能性があるため timeout=None
                child.expect(r'Which trial do you want to use\?', timeout=None)
                print(f"[auto-select] 最適化完了。Trial [{trial_selection}] を選択します...")
                child.sendline(str(trial_selection))

                # 3. 保存やチャットのメニューが出た場合の処理
                if auto_exit_after_trial:
                    try:
                        child.expect(r'What do you want to do with the decensored model\?', timeout=300)
                        child.expect(r'Enter number:', timeout=60)
                        print("[auto-select] モデル操作メニューを検出。保存を選択します...")
                        child.sendline('1')  # [1] Save the model to a local folder

                        child.expect(r'Path to the folder:', timeout=120)
                        child.sendline(LOCAL_EXPORTED_MODEL)
                        print(f"[auto-select] 保存先ディレクトリを設定: {LOCAL_EXPORTED_MODEL}")

                        # 量子化時は保存戦略の選択が追加で出る
                        try:
                            child.expect(r'How do you want to proceed\?', timeout=120)
                            child.expect(r'Enter number:', timeout=60)
                            child.sendline('2')  # Save LoRA adapter only
                            print("[auto-select] 保存戦略を選択: adapter")
                        except (pexpect.TIMEOUT, pexpect.EOF):
                            print("[auto-select] 保存戦略プロンプトは表示されませんでした。")

                        child.expect(r'Model saved to', timeout=None)
                        print("[auto-select] モデル保存が完了。Trial メニューに戻ります...")

                        child.expect(r'What do you want to do with the decensored model\?', timeout=120)
                        child.expect(r'Enter number:', timeout=60)
                        child.sendline('5')  # Return to the trial selection menu

                        child.expect(r'Which trial do you want to use\?', timeout=120)
                        child.expect(r'Enter number:', timeout=60)
                        child.sendline('12')  # Exit program
                        print("[auto-select] Trial 選択メニューからプログラムを終了します。")
                    except (pexpect.TIMEOUT, pexpect.EOF):
                        print("[auto-select] メニューが見つからないか、プロセスが既に終了しました。")

            except pexpect.TIMEOUT:
                print("[auto-select] Trial 選択メニュー待機中にタイムアウトしました。")
            except pexpect.EOF:
                print("[auto-select] Trial 選択メニュー前にプロセスが終了しました。")

            # プロセス終了を待つ
            try:
                child.expect(pexpect.EOF)
            except:
                pass

            result_code = child.exitstatus if child.exitstatus is not None else 0
    except Exception as e:
        print(f"[auto-resume error] {e}。subprocess.run にフォールバックします...")
        result = subprocess.run(
            f"{cmd} 2>&1 | tee {log_file}",
            shell=True,
            check=False
        )
        result_code = result.returncode
else:
    print("=== 標準 subprocess で Heretic を開始 ===")
    result = subprocess.run(
        f"{cmd} 2>&1 | tee {log_file}",
        shell=True,
        check=False
    )
    result_code = result.returncode

print()
print("=== Drive へ最終同期 ===")
os.makedirs(DRIVE_CKPTS, exist_ok=True)
subprocess.run(['rsync', '-av', '--progress', f'{LOCAL_CKPTS}/', f'{DRIVE_CKPTS}/'], check=False)
print(f"最終同期が完了しました。Drive のチェックポイント保存先: {DRIVE_CKPTS}")

if os.path.isdir(LOCAL_EXPORTED_MODEL):
    os.makedirs(DRIVE_EXPORTED_MODEL, exist_ok=True)
    subprocess.run([
        'rsync', '-av', '--progress',
        f'{LOCAL_EXPORTED_MODEL}/',
        f'{DRIVE_EXPORTED_MODEL}/'
    ], check=False)
    print(f"最終同期が完了しました。Drive のエクスポートモデル保存先: {DRIVE_EXPORTED_MODEL}")
else:
    print("エクスポートモデルのフォルダが見つからないため、モデル同期をスキップします。")
print()
print(f"終了コード: {result_code}")
if result_code == 0:
    print("✓ Heretic が正常に完了しました")
else:
    print("✗ Heretic がエラーで終了しました（上のログを確認してください）")